

--- CREADO POR: GABRIELA MICHELLE GARCIA PEREZ

--- FECHA: 15/10/2026



In [1]:
#INSTALAR LIBRERIAS
!pip install jinja2 weasyprint python-docx -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.9/328.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.4/829.4 kB 27.4 MB/s eta 0:00:00


In [2]:
from pathlib import Path
from google.colab import files

BASE = Path('/content/EP2_Parte3_Gabriela_Garcia')
BASE.mkdir(parents=True, exist_ok=True)

print("Subir operacion_sedes.csv")
uploaded = files.upload()
for nombre, contenido in uploaded.items():
    (BASE / nombre).write_bytes(contenido)
    print(f"  {nombre} → {BASE}")

Subir operacion_sedes.csv


Saving operacion_sedes.csv to operacion_sedes.csv
  operacion_sedes.csv → /content/EP2_Parte3_Gabriela_Garcia


In [5]:
# IMPORTAR LIBRERÍAS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from jinja2 import Template
import base64
import json
import hashlib

# CONFIGURAR MATPLOTLIB
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print(" Librerías importadas")
print(f"   - pandas: {pd.__version__}")
print(f"   - matplotlib: {plt.matplotlib.__version__}")

 Librerías importadas
   - pandas: 2.2.3
   - matplotlib: 3.10.0


In [7]:
# CARGAR DATASET
ruta_fuente = BASE/'operacion_sedes.csv'
df = pd.read_csv(ruta_fuente)
df['fecha'] = pd.to_datetime(df['fecha'])

print(f"Registros: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Periodo: {df['fecha'].min().date()} a {df['fecha'].max().date()}") #esto es para verificar el periodo de tiempo en que se encuentra el archivo
print(df.head(3).to_string())

Registros: 1620 filas x 6 columnas
Periodo: 2026-01-01 a 2026-06-29
       fecha sede_id     region  generacion_kwh  demanda_kwh  evento_programado
0 2026-01-01     S01  Occidente         1564.65      1645.95                  1
1 2026-01-01     S02  Occidente         1831.21      1714.67                  1
2 2026-01-01     S03  Occidente         1998.01      1768.40                  1


In [9]:
# CALULAR LOS KPIS
# CALCULAMOS EL DEFICIT DE ENERGIA EN CADA SEDE
df['deficit_kwh'] = (df['demanda_kwh']-df['generacion_kwh']).clip(lower=0)
# utilizo .clip para que se vuelvan cero los negativos sugiriendonos que no hubo
# deficit de energia

kpis = {
    'energia_total_kwh': df['generacion_kwh'].sum(),
    'demanda_total_kwh': df['demanda_kwh'].sum(),
    'deficit_total_kwh': df['deficit_kwh'].sum(),
    'periodo': f"{df['fecha'].min().strftime('%d/%m/%Y')} - {df['fecha'].max().strftime('%d/%m/%Y')}",
}

print("KPIs DEL PERIODO")
print("=" * 50)
print(f"Energía generada:  {kpis['energia_total_kwh']:>14,.2f} kWh")
print(f"Demanda total:     {kpis['demanda_total_kwh']:>14,.2f} kWh")
print(f"Déficit total:     {kpis['deficit_total_kwh']:>14,.2f} kWh")
print(f"Periodo:           {kpis['periodo']}")
print("=" * 50)

KPIs DEL PERIODO
Energía generada:    2,913,623.55 kWh
Demanda total:       3,146,866.67 kWh
Déficit total:         286,904.20 kWh
Periodo:           01/01/2026 - 29/06/2026


In [14]:
# ANALISIS POR REGION
df_region = df.groupby('region').agg(
    energia_kwh= ('generacion_kwh', 'sum'),
    demanda_kwh= ('demanda_kwh', 'sum'),
    deficit_kwh= ('deficit_kwh', 'sum'),
).reset_index()

# Ordenar de mayor a menor déficit (igual que sort_values de la Guía 07)
df_region = df_region.sort_values('deficit_kwh', ascending=False)

df_sede = df.groupby('sede_id').agg(
    deficit_kwh= ('deficit_kwh', 'sum'),
).reset_index().sort_values('deficit_kwh', ascending=False)

print("ANÁLISIS POR SEDE:")
print(df_sede.to_string(index=False))

print("\nANÁLISIS POR REGIÓN:")
print(df_region.to_string(index=False))

print("\nSuma del déficit por región:", round(df_region['deficit_kwh'].sum(), 2))
print("Suma del déficit por sede:  ", round(df_sede['deficit_kwh'].sum(), 2))
print("Déficit total (KPI):        ", kpis['deficit_total_kwh'])

ANÁLISIS POR SEDE:
sede_id  deficit_kwh
    S04     77336.41
    S07     69364.49
    S09     42266.81
    S01     36692.27
    S05     25528.84
    S02     20465.48
    S06     10909.28
    S08      3593.47
    S03       747.15

ANÁLISIS POR REGIÓN:
   region  energia_kwh  demanda_kwh  deficit_kwh
  Oriente   1087683.86   1185888.56    115224.77
   Centro    945327.20   1050519.18    113774.53
Occidente    880612.49    910458.93     57904.90

Suma del déficit por región: 286904.2
Suma del déficit por sede:   286904.2
Déficit total (KPI):         286904.19999999995


In [19]:
# ===================================================================
# GRÁFICOS PARA EL INFORME
# ===================================================================

# GRÁFICO 1: Déficit por sede
fig, ax = plt.subplots(figsize=(10, 6))
colores = plt.cm.Blues(np.linspace(0.9, 0.4, len(df_sede)))   # más oscuro = más déficit

# Barras: eje X = sede, eje Y = déficit
barras = ax.bar(df_sede['sede_id'], df_sede['deficit_kwh'],
                color=colores, edgecolor='#1a5490', linewidth=1.5)

ax.set_title('Déficit energético por sede', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Sede', fontsize=11)
ax.set_ylabel('Déficit (kWh)', fontsize=11)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

for barra, valor in zip(barras, df_sede['deficit_kwh']):
    height = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., height,
            f'{valor:,.0f}', ha='center', va='bottom',
            fontsize=10, fontweight='bold')

plt.tight_layout()
ruta_grafico = BASE / 'grafico.png'
plt.savefig(ruta_grafico, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"Gráfico 1 (informe): {ruta_grafico.name}")


# GRÁFICO 2: Déficit por región
fig, ax = plt.subplots(figsize=(10, 6))
colores = plt.cm.Blues(np.linspace(0.9, 0.4, len(df_region)))

# Barras: eje X = región, eje Y = déficit
barras = ax.bar(df_region['region'], df_region['deficit_kwh'],
                color=colores, edgecolor='#1a5490', linewidth=1.5)

ax.set_title('Déficit energético por región', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Región', fontsize=11)
ax.set_ylabel('Déficit (kWh)', fontsize=11)
ax.set_ylim(0, df_region['deficit_kwh'].max() * 1.1)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

for barra, valor in zip(barras, df_region['deficit_kwh']):
    height = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., height,
            f'{valor:,.0f}', ha='center', va='bottom',
            fontsize=10, fontweight='bold')

plt.tight_layout()
ruta_grafico_region = BASE / 'grafico_region.png'
plt.savefig(ruta_grafico_region, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"Gráfico 2 (extra):   {ruta_grafico_region.name}")

Gráfico 1 (informe): grafico.png
Gráfico 2 (extra):   grafico_region.png


In [25]:
# INSTALAR HARFBUZZ-SUBSET
!apt-get update -qq
!apt-get install -y libharfbuzz-subset0 -qq

# Reinstalar WeasyPrint con soporte para HarfBuzz
!pip install -U weasyprint -q

print("HarfBuzz-Subset instalado")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libharfbuzz-subset0:amd64.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../libharfbuzz-subset0_8.3.0-2build2_amd64.deb ...
Unpacking libharfbuzz-subset0:amd64 (8.3.0-2build2) ...
Setting up libharfbuzz-subset0:amd64 (8.3.0-2build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/loc

In [27]:
!pip install -U weasyprint -q

In [20]:
# ===================================================================
# PLANTILLA JINJA2 (Guía 07: "Template HTML con Jinja2")
# ===================================================================
template_informe = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Informe Gerencial - GreenEnergy Analytics</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 40px; color: #333; line-height: 1.6; }
        h1 { color: #1a5490; border-bottom: 3px solid #1a5490; padding-bottom: 10px; }
        h2 { color: #2c7bb6; margin-top: 30px; border-bottom: 1px solid #ddd; padding-bottom: 5px; }
        .kpi-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 15px; margin: 20px 0; }
        .kpi-card { background: #f0f7ff; padding: 15px; border-left: 4px solid #2c7bb6; border-radius: 4px; }
        .kpi-valor { font-size: 22px; font-weight: bold; color: #1a5490; }
        .kpi-label { font-size: 12px; color: #666; text-transform: uppercase; }
        .hallazgo { background: #f9f9f9; padding: 15px; border-left: 4px solid #ff9800; margin: 15px 0; }
        .limitacion { background: #fff3cd; padding: 10px; border-left: 4px solid #ffc107; margin: 10px 0; }
        img { max-width: 100%; border: 1px solid #eee; }
        .pie { font-size: 11px; color: #999; margin-top: 40px; border-top: 1px solid #ddd; padding-top: 10px; text-align: center; }
    </style>
</head>
<body>

<h1>INFORME GERENCIAL DE COBERTURA ENERGÉTICA</h1>
<p><strong>Periodo:</strong> {{ periodo }}</p>
<p><strong>Fecha de generación:</strong> {{ fecha_generacion }}</p>

<h2>1. Indicadores principales</h2>
<div class="kpi-grid">
    <div class="kpi-card">
        <div class="kpi-label">Energía generada (kWh)</div>
        <div class="kpi-valor">{{ "{:,.2f}".format(kpis.energia_total_kwh) }}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Demanda total (kWh)</div>
        <div class="kpi-valor">{{ "{:,.2f}".format(kpis.demanda_total_kwh) }}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Deficit total (kWh)</div>
        <div class="kpi-valor">{{ "{:,.2f}".format(kpis.deficit_total_kwh) }}</div>
    </div>
</div>

<h2>2. Déficit por sede</h2>
<img src="{{ ruta_grafico }}" alt="Déficit por sede">

<h2>3. Hallazgo</h2>
<div class="hallazgo">{{ hallazgo }}</div>

<h2>4. Recomendación</h2>
<p>{{ recomendacion }}</p>

<h2>5. Limitación</h2>
<div class="limitacion">{{ limitacion }}</div>

<div class="pie">Informe generado automáticamente por generar_reporte.py</div>

</body>
</html>
"""

# Guardarla como archivo (el enunciado exige el nombre plantilla.html)
(BASE / 'plantilla.html').write_text(template_informe, encoding='utf-8')
print("Plantilla guardada: plantilla.html")

Plantilla guardada: plantilla.html


In [33]:

# DATOS DEL TEMPLATE Y RENDER A HTML

# FUNCIÓN PARA CONVERTIR IMAGEN A BASE64
def imagen_a_base64(ruta):
    with open(ruta, 'rb') as f:
        return f"data:image/png;base64,{base64.b64encode(f.read()).decode()}"

# Variables calculadas a partir mis datos
sede_1 = df_sede.iloc[0]['sede_id']
deficit_sede_1 = df_sede.iloc[0]['deficit_kwh']
top2_kwh = df_sede.head(2)['deficit_kwh'].sum()
top2_pct = top2_kwh / kpis['deficit_total_kwh'] * 100
sede_2 = df_sede.iloc[1]['sede_id']
deficit_sede_2 = df_sede.iloc[1]['deficit_kwh']

hallazgo = (
    f"Las sedes {sede_1} ({deficit_sede_1:,.0f} kWh) y {sede_2} ({deficit_sede_2:,.0f} kWh) "
    f"concentran {top2_pct:.1f}% del déficit total del periodo {kpis['periodo']} "
    f"({kpis['deficit_total_kwh']:,.0f} kWh). "
    f"Esto describe dónde se ubica el déficit, no su causa."
)
recomendacion = (
    f"Investigar las sedes {sede_1} y {sede_2}, que concentran el {top2_pct:.1f}% del déficit de energía, "
    f"para identificar por qué su generación no alcanza la demanda de estas sedes."
)

limitacion = (
    "El análisis del déficit de energía comprende un periodo de 180 días, por lo que únicamente "
    "sirve como prototipo, no para tomar decisiones reales. Además, el análisis solo muestra dónde "
    "se genera el déficit de energía, no sus posibles causas."
)

datos_template = {
    'periodo': kpis['periodo'],
    'fecha_generacion': datetime.now().strftime('%d/%m/%Y %H:%M'),
    'kpis': kpis,
    'ruta_grafico': imagen_a_base64(ruta_grafico),
    'hallazgo': hallazgo,
    'recomendacion': recomendacion,
    'limitacion': limitacion,
}

# Renderizar
template = Template((BASE / 'plantilla.html').read_text(encoding='utf-8'))
html_renderizado = template.render(**datos_template)

ruta_html = BASE / 'informe.html'
ruta_html.write_text(html_renderizado, encoding='utf-8')
print(f"HTML generado: {ruta_html.name}")

# convertir a pdf
from weasyprint import HTML as WeasyHTML

ruta_pdf = BASE / 'informe.pdf'
WeasyHTML(string=html_renderizado).write_pdf(str(ruta_pdf))

print(f"PDF generado: {ruta_pdf.name}")
print(f"Tamaño: {ruta_pdf.stat().st_size / 1024:.1f} KB")

HTML generado: informe.html


PDF generado: informe.pdf
Tamaño: 72.4 KB


In [32]:
# reporte_kpis.csv Y metadatos.json (Guia: "Metadata de trazabilidad")

#reporte_kpis.csv con indicador y valor para los kpis
df_kpis = pd.DataFrame({
    'indicador': ['energia_total_kwh', 'demanda_total_kwh', 'deficit_total_kwh'],
    'valor': [round(kpis['energia_total_kwh'], 2),
              round(kpis['demanda_total_kwh'], 2),
              round(kpis['deficit_total_kwh'], 2)]
})
ruta_kpis = BASE / 'reporte_kpis.csv'
df_kpis.to_csv(ruta_kpis, index=False)
print(df_kpis.to_string(index=False))

# metadatos.json: fuente, registros_entrada, sha256_fuente
sha256_fuente = hashlib.sha256(ruta_fuente.read_bytes()).hexdigest()

metadatos = {
    'fuente': 'operacion_sedes.csv',
    'registros_entrada': len(df),
    'sha256_fuente': sha256_fuente,
}
ruta_meta = BASE / 'metadatos.json'
ruta_meta.write_text(json.dumps(metadatos, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(metadatos, indent=2, ensure_ascii=False))

        indicador      valor
energia_total_kwh 2913623.55
demanda_total_kwh 3146866.67
deficit_total_kwh  286904.20
{
  "fuente": "operacion_sedes.csv",
  "registros_entrada": 1620,
  "sha256_fuente": "9c28479ba39e4204b20bacdb23341e155ad1b9d8d517338906ae540b005520cb"
}
